# Foundation Strategy: Return Predictability + Volatility Timing
### Applying Lectures 5.6, 5.7, and 11 to Kalshi Prediction Markets

This notebook replicates the full walk-forward backtest of the Foundation Strategy and
evaluates it using the **performance evaluation framework from Lecture 11**, including:
alpha testing, appraisal ratios, SR standard errors, fraction-to-half, sample splitting,
multiple testing adjustment, and a data-mining discussion.

**Strategy summary:**  
Two techniques from lecture notes are applied to Kalshi prediction markets:
1. **Return Predictability (Lecture 5.6):** OLS slope of recent price changes estimates expected return (μ̂).
2. **Volatility Timing (Lecture 5.7):** Position scaled by inverse realized variance (1/RV).

Combined mean-variance optimal weight: $w = \hat{\mu} / (\gamma \cdot RV)$

**Data:** Live from the Kalshi public API — no file upload needed.  
**Runtime:** ~5–8 minutes to fetch 180 days of history for 200 markets.

In [ ]:
!pip install -q requests pandas scipy numpy tabulate matplotlib statsmodels pandas-datareader

In [ ]:
import time
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
import pandas_datareader.data as web
from scipy import stats
from scipy.stats import norm
from tabulate import tabulate

warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('Imports OK')

In [ ]:
# ── Strategy parameters ───────────────────────────────────────────────────────
GAMMA       = 2.0    # Risk aversion γ
LOOKBACK    = 30     # Days of price history used for OLS + RV estimation
MIN_R2      = 0.10   # Minimum OLS R² to trust the signal
MAX_WEIGHT  = 0.25   # Cap on raw weight before normalization
MIN_HISTORY = 10     # Minimum data points required to run estimate()
MAX_SPREAD  = 0.10   # Skip markets with bid-ask spread > 10% of mid-price
MIN_PRICE   = 0.15   # Skip trades where entry price < 15% (P<0.15 had SR=-0.34 empirically)
HOLD_PERIOD = 7      # Hold period in days
FETCH_DAYS  = 180    # Days of price history to fetch per market
LIMIT       = 200    # Max markets to process, ranked by volume

BASE_URL = 'https://api.elections.kalshi.com/trade-api/v2'
HEADERS  = {'Accept': 'application/json'}

SPORTS_KEYWORDS = [
    'nba', 'nfl', 'nhl', 'mlb', 'nascar', 'pga', 'premier',
    'champions', 'bundesliga', 'laliga', 'seriea', 'ligue1',
    'mvp', 'championship', 'playoff', 'superbowl', 'worldseries'
]

print('Configuration set.')

## 1. Data Access — Kalshi Prediction Market API

The notebook version of `fetch_market_history` returns *both* prices and calendar dates so
that we can build a proper daily return time-series for the CAPM alpha test in Section 6.

In [ ]:
def kalshi_get(path, params={}):
    resp = requests.get(f"{BASE_URL}/{path.lstrip('/')}",
                        headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def to_float(val):
    """Parse to float and validate it is a probability in [0, 1]."""
    try:
        f = float(val)
        return f if 0.0 <= f <= 1.0 else None
    except (TypeError, ValueError):
        return None


def fetch_events():
    """Page through all open Kalshi events across all categories."""
    events, cursor, page = [], None, 0
    print('Fetching all open events…', end='', flush=True)
    while True:
        params = {'limit': 100, 'status': 'open', 'with_nested_markets': 'true'}
        if cursor:
            params['cursor'] = cursor
        data   = kalshi_get('events', params)
        batch  = data.get('events', [])
        events.extend(batch)
        page  += 1
        cursor = data.get('cursor')
        print(f'\r  Page {page}: {len(events)} events…', end='', flush=True)
        if not cursor or not batch:
            break
        time.sleep(0.2)
    print(f'\r  Done — {len(events)} events fetched.          ')
    return events


def extract_markets(events):
    markets = []
    for ev in events:
        for m in ev.get('markets', []):
            bid  = to_float(m.get('yes_bid_dollars'))
            ask  = to_float(m.get('yes_ask_dollars'))
            last = to_float(m.get('last_price_dollars'))
            vol  = float(m.get('volume_fp') or 0)
            if vol > 0 and (bid or ask or last):
                m['_event_title'] = ev.get('title', '')
                markets.append(m)
    return markets


def fetch_market_history(ticker, event_ticker, days):
    """
    Return (dates, prices) — two aligned lists.
    dates: list of pd.Timestamp (one per daily candle)
    prices: list of float close prices in (0, 1)
    Dates allow building a calendar-indexed return series for the CAPM test.
    """
    try:
        series_ticker = event_ticker.split('-')[0] if event_ticker else ''
        if not series_ticker:
            return [], []
        end_ts   = int(time.time())
        start_ts = end_ts - 86400 * days
        data = kalshi_get(
            f'series/{series_ticker}/markets/{ticker}/candlesticks',
            params={'start_ts': start_ts, 'end_ts': end_ts, 'period_interval': 1440},
        )
        prices, dates = [], []
        for c in data.get('candlesticks', []):
            close = to_float((c.get('price') or {}).get('close_dollars'))
            ts    = c.get('end_ts') or c.get('start_ts')
            if close is not None and 0 < close < 1:
                prices.append(close)
                dates.append(pd.Timestamp(ts, unit='s').normalize() if ts else None)
        return dates, prices
    except Exception:
        return [], []


def spread_pct(m):
    bid = to_float(m.get('yes_bid_dollars'))
    ask = to_float(m.get('yes_ask_dollars'))
    if not bid or not ask or ask == 0:
        return 0.0
    mid = (bid + ask) / 2
    return (ask - bid) / mid if mid else 0.0


print('API helpers defined.')

## 2. Signal Estimation (Lectures 5.6 + 5.7)

**μ̂** — OLS slope of daily price changes, negated (mean-reversion, not momentum).  
**RV** — Realized variance of daily price changes, annualized × 252.  
**R² filter** — OLS R² ≥ 0.10 required to trade the signal.

In [ ]:
def estimate(prices):
    n = len(prices)
    if n < MIN_HISTORY:
        return None
    changes = np.diff(np.array(prices, dtype=float))
    if len(changes) < 2:
        return None
    x = np.arange(len(changes), dtype=float)
    slope, _, r_val, _, _ = stats.linregress(x, changes)
    r2 = r_val ** 2
    rv = float(np.var(changes, ddof=1)) * 252
    if rv < 1e-8:
        return None
    return {'mu_hat': -slope * 252, 'rv': rv, 'r2': r2, 'n': n}


def mv_weight(mu_hat, rv, gamma, max_weight):
    """w = μ̂ / (γ · RV), capped at ±max_weight."""
    return float(np.clip(mu_hat / (gamma * rv), -max_weight, max_weight))


print('Signal functions defined.')

## 3. Walk-Forward Backtest Engine

P&L per dollar of bankroll — three cost layers:
$$\text{net PnL} = w(p_{t+h}-p_t) \;-\; \underbrace{|w|\cdot\tfrac{\text{spread}}{2}\cdot p_t}_{\text{spread}} \;-\; \underbrace{0.07\cdot|w|\cdot p_t}_{\text{Kalshi fee}}$$

**Kalshi fee derivation:** $0.07\times C\times P\times(1-P)$ with $C=|w|/(1-P)$ for a BUY NO trade simplifies to $0.07\times|w|\times P$.

In [ ]:
def simulate_market(ticker, title, dates, prices, sprd,
                    lookback, hold_period, gamma, max_weight, min_r2, min_price):
    """
    Walk forward through a single market's price history.
    Returns a list of trade dicts, including entry_date for time-series alignment.
    """
    trades = []
    n = len(prices)

    for t in range(lookback, n - hold_period):
        hist = prices[t - lookback : t]
        est  = estimate(hist)
        if est is None or est['r2'] < min_r2:
            continue

        w = mv_weight(est['mu_hat'], est['rv'], gamma, max_weight)
        if w >= 0:   # BUY YES excluded — mean-reversion signal is BUY NO only
            continue

        entry = prices[t]
        if entry < min_price:   # near-floor markets have negative realized Sharpe
            continue

        exit_ = prices[t + hold_period]

        raw_pnl    = w * (exit_ - entry)
        spread_cost = abs(w) * sprd * entry / 2
        kalshi_fee  = 0.07 * abs(w) * entry   # 0.07×C×P×(1-P), simplified for BUY NO
        net_pnl     = raw_pnl - spread_cost - kalshi_fee

        trades.append({
            'Ticker':     ticker,
            'Title':      title[:45],
            'Entry Date': dates[t] if dates and t < len(dates) else None,
            'Entry P':    round(entry, 4),
            'Exit P':     round(exit_, 4),
            'Weight':     round(w, 4),
            'mu_hat':     round(est['mu_hat'], 4),
            'RV':         round(est['rv'], 4),
            'R2':         round(est['r2'], 3),
            'Raw PnL':    round(raw_pnl, 6),
            'Net PnL':    round(net_pnl, 6),
            'Kalshi Fee': round(kalshi_fee, 6),
            'Spread':     round(sprd, 4),
            'Win':        net_pnl > 0,
        })

    return trades


print('Backtest engine defined.')

## 4. Run the Backtest

Parameters: γ=2, lookback=30d, hold=7d, fetch=180d, min-R²=0.10, min-price=0.15, limit=200.

**Note:** This cell fetches live API data — runtime ~5–8 minutes.

In [ ]:
events  = fetch_events()
markets = extract_markets(events)
markets = [m for m in markets if float(m.get('volume_fp') or 0) >= 50]
markets = [m for m in markets if not any(
    kw in (m.get('ticker') or '').lower() or kw in (m.get('title') or '').lower()
    for kw in SPORTS_KEYWORDS
)]
markets.sort(key=lambda m: float(m.get('volume_fp') or 0), reverse=True)
markets = markets[:LIMIT]
print(f'Markets after filters: {len(markets)}')

all_trades = []
for i, m in enumerate(markets):
    ticker       = m.get('ticker', '')
    event_ticker = m.get('event_ticker', '')
    title        = m.get('title') or ticker
    sprd         = spread_pct(m)
    if sprd > MAX_SPREAD:
        continue
    dates, prices = fetch_market_history(ticker, event_ticker, days=FETCH_DAYS)
    time.sleep(0.15)
    if len(prices) < LOOKBACK + HOLD_PERIOD + 1:
        continue
    trades = simulate_market(
        ticker=ticker, title=title, dates=dates, prices=prices, sprd=sprd,
        lookback=LOOKBACK, hold_period=HOLD_PERIOD, gamma=GAMMA,
        max_weight=MAX_WEIGHT, min_r2=MIN_R2, min_price=MIN_PRICE,
    )
    all_trades.extend(trades)
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(markets)} markets — {len(all_trades)} trades')

df = pd.DataFrame(all_trades)
print(f'\nDone: {len(df)} trades across {df["Ticker"].nunique()} markets.')

## 5. Backtest Results — Summary Table

In [ ]:
net        = df['Net PnL']
raw        = df['Raw PnL']
n_trades   = len(df)
ann_factor = np.sqrt(252 / HOLD_PERIOD)
mean_raw   = raw.mean()
mean_net   = net.mean()
std_net    = net.std(ddof=1)
sharpe_raw = (raw.mean() / raw.std(ddof=1)) * ann_factor if raw.std(ddof=1) > 1e-8 else 0.0
sharpe_net = (mean_net / std_net) * ann_factor if std_net > 1e-8 else 0.0
cum        = net.cumsum()
max_dd     = (cum - cum.cummax()).min()
pf         = net[net > 0].sum() / net[net < 0].abs().sum()

summary = [
    ['Trades (n)',               n_trades],
    ['Markets',                  df['Ticker'].nunique()],
    ['Hold period',              f'{HOLD_PERIOD}d'],
    ['Win rate',                 f"{df['Win'].mean()*100:.1f}%"],
    ['Mean raw P&L / trade',     f'{mean_raw:+.6f}'],
    ['Mean net P&L / trade',     f'{mean_net:+.6f}'],
    ['Profit factor',            f'{pf:.2f}x'],
    ['Max cumulative drawdown',  f'{max_dd:+.6f}'],
    ['Annualized Sharpe (raw)',  f'{sharpe_raw:+.3f}'],
    ['Annualized Sharpe (net)',  f'{sharpe_net:+.3f}'],
]
print(tabulate(summary, headers=['Metric', 'Value'], tablefmt='rounded_outline'))

---
## 6. Performance Evaluation (Lecture 11 Framework)

Following the Lecture 11 diagnostics toolkit exactly:

> *"You should look at many things when evaluating a strategy: Sharpe + t-stat, alpha/t-alpha/appraisal ratio,
> cumulative return and drawdown plots, tail behavior, fraction to half, and comparison to a benchmark."*

### 6.1 Diagnostics Functions

Three helper functions from Lecture 11, adapted for trade-level returns:
- **`SR_vol(R)`** — analytical Sharpe ratio standard error (assumes normality)
- **`SR_vol_boot(R)`** — bootstrap Sharpe SE (distribution-free)
- **`fractiontohalf(R)`** — greedy fraction-to-half test

In [ ]:
def SR_vol(R):
    """
    Analytical Sharpe ratio standard error (Lecture 11).
    Assumes i.i.d. normally distributed returns.
    SE(SR) = sqrt[ (1 + SR²/2) / (T-1) ]
    """
    SR = R.mean() / R.std()
    T  = len(R)
    return np.sqrt((1 + SR**2 / 2) / (T - 1))


def SR_vol_boot(R, N=2000, seed=42):
    """
    Bootstrap Sharpe ratio standard error (Lecture 11).
    Resamples with replacement N times; returns (std, 5th-pct).
    No normality assumption — better for fat-tailed returns.
    """
    rng = np.random.default_rng(seed)
    T   = len(R)
    sr_boot = []
    for _ in range(N):
        samp = rng.choice(R.values, size=T, replace=True)
        sd   = samp.std(ddof=1)
        if sd > 1e-10:
            sr_boot.append(samp.mean() / sd)
    sr_boot = np.array(sr_boot)
    return sr_boot.std(), np.percentile(sr_boot, 5)


def fractiontohalf(R):
    """
    Greedy fraction-to-half (Lecture 11).
    Removes the highest return one at a time until SR falls below SR_original/2.
    Returns the fraction of observations removed.

    Interpretation: a small fraction (e.g. 2-3%) means a few extreme
    observations are carrying the entire Sharpe ratio — fragile performance.
    """
    R_work = R.copy()
    target = R_work.mean() / R_work.std() / 2   # half of original SR
    order  = R_work.sort_values(ascending=False).index
    removed = 0
    for idx in order:
        if len(R_work) <= 2:
            break
        current_sr = R_work.mean() / R_work.std()
        if current_sr <= target:
            break
        R_work  = R_work.drop(idx)
        removed += 1
    return removed / len(R)


print('Diagnostic helper functions defined.')

### 6.2 The Alpha Test

From Lecture 11: *"Subtract the risk-free rate from raw returns, choose a factor model (we start with CAPM), run the regression $R_t - R_f = \alpha + \beta(R_m - R_f) + \varepsilon_t$, and test whether $\alpha$ exceeds your hurdle."*

**Adaptation for prediction markets:**  
Prediction market contracts are binary bets on political and economic events — they have **no structural equity beta**. Buying a NO contract on a Senate race is unrelated to S&P 500 returns. We verify this empirically by building a daily return series and regressing on Mkt-RF.

Since $\beta \approx 0$, the CAPM reduces to: $\alpha = \bar{R} - R_f \approx \bar{R}$ (RF over a 7-day hold is ~0.01%). The alpha test becomes a direct test of mean net P&L > 0.

The **appraisal ratio** is:  $AR = \alpha / \sigma_\varepsilon$
where $\sigma_\varepsilon$ is the residual volatility from the factor regression.  
For a $\beta \approx 0$ strategy: $AR \approx SR$ (the two coincide).

In [ ]:
# Build a daily P&L time-series from trade entry dates
df_dated = df.dropna(subset=['Entry Date'])
daily_pnl = df_dated.groupby('Entry Date')['Net PnL'].sum().rename('Strategy')
daily_pnl.index = pd.to_datetime(daily_pnl.index)

print(f'Date range of trades: {daily_pnl.index.min().date()} to {daily_pnl.index.max().date()}')
print(f'Trading days with at least one entry: {len(daily_pnl)}')

# Fetch Fama-French daily factors for the same window
try:
    ff = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench',
                        start=daily_pnl.index.min() - pd.Timedelta(days=10))[0] / 100
    ff.index = pd.to_datetime(ff.index)
    merged = daily_pnl.to_frame().join(ff[['Mkt-RF', 'RF']], how='inner')
    print(f'Merged with FF daily factors: {len(merged)} overlapping days')
    has_ff = len(merged) >= 5
    if not has_ff:
        print('Insufficient overlap with FF factors — running alpha test without factor alignment.')
        merged = daily_pnl.to_frame()
        merged['Mkt-RF'] = 0.0
        merged['RF']     = 0.0
except Exception as e:
    print(f'FF data unavailable ({e}) — running alpha test without factor alignment.')
    merged = daily_pnl.to_frame()
    merged['Mkt-RF'] = 0.0
    merged['RF']     = 0.0
    has_ff = False

In [ ]:
# CAPM alpha regression: (Strategy - RF) = α + β·(Mkt-RF) + ε
y = merged['Strategy'] - merged['RF']
x = sm.add_constant(merged['Mkt-RF'])
capm = sm.OLS(y, x).fit()

alpha_daily  = capm.params['const']
beta         = capm.params['Mkt-RF']
t_alpha      = capm.tvalues['const']
t_beta       = capm.tvalues['Mkt-RF']
resid_vol    = capm.resid.std()
ar_daily     = alpha_daily / resid_vol if resid_vol > 1e-10 else 0.0

# Annualise (252 trading days / HOLD_PERIOD day trades → effective periods/year)
alpha_ann  = alpha_daily * (252 / HOLD_PERIOD)
ar_ann     = ar_daily * np.sqrt(252 / HOLD_PERIOD)

capm_rows = [
    ['α (daily)',              f'{alpha_daily:+.6f}'],
    ['α (annualised)',         f'{alpha_ann:+.4f}'],
    ['t-stat(α)',              f'{t_alpha:+.3f}'],
    ['β (market)',             f'{beta:+.4f}  ← expected ≈ 0 for prediction markets'],
    ['t-stat(β)',              f'{t_beta:+.3f}'],
    ['Residual vol (daily)',   f'{resid_vol:.6f}'],
    ['Appraisal ratio (AR)',   f'{ar_ann:+.3f}  (annualised: α / σ_ε)'],
    ['R² of CAPM regression', f'{capm.rsquared:.4f}'],
]
print(tabulate(capm_rows, headers=['Metric', 'Value'], tablefmt='rounded_outline'))

if has_ff:
    print('\nFull OLS summary:')
    print(capm.summary())

### 6.3 The Pod Manager Decision: When to Accept or Discard a Strategy

From Lecture 11: you want to be $p$-confident that the appraisal ratio exceeds target $AR^*$.
The minimum *observed* AR that clears the bar given $T$ years of data is:
$$AR_{\min}(T) = AR^* + z_p / \sqrt{T}$$

The threshold is high when $T$ is small (high uncertainty) and shrinks as evidence accumulates.

In [ ]:
def plot_alpha_threshold(ar_target, T_max_months, p, title_extra=''):
    """Minimum observed AR needed to be p-confident the true AR ≥ ar_target."""
    T = np.arange(1, T_max_months + 1)
    z = norm.ppf(p)
    threshold = ar_target + z / np.sqrt(T / 12)

    # Mark where our strategy currently sits
    T_ours = len(df) / (252 / HOLD_PERIOD) / 12  # approx years equivalent
    thresh_ours = ar_target + z / np.sqrt(T_ours)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(T / 12, threshold, linewidth=2, label='Min observed AR to accept')
    ax.axhline(ar_ann, color='red', linestyle='--', linewidth=1.5,
               label=f'Our observed AR = {ar_ann:.2f}')
    ax.axvline(T_ours, color='grey', linestyle=':', linewidth=1,
               label=f'Our sample ≈ {T_ours:.2f} yr equiv.')
    ax.set_xlabel('Years of data')
    ax.set_ylabel('Appraisal ratio threshold')
    ax.set_title(f'Min observed AR to be {p:.0%} confident true AR ≥ {ar_target}{title_extra}')
    ax.legend()
    plt.tight_layout()
    plt.show()

# When to accept: p=70% confident AR > 1
plot_alpha_threshold(ar_target=1.0, T_max_months=5*12, p=0.70,
                     title_extra=' (accept threshold)')

# When to discard: p=75% confident AR < 1 (i.e. p=25% accept)
plot_alpha_threshold(ar_target=1.0, T_max_months=5*12, p=0.25,
                     title_extra=' (discard threshold)')

### 6.4 Complete Diagnostics Suite

Adapted from the Lecture 11 `Diagnostics` function. Computes:
SR, SR factor, vol, mean return, t-mean, alpha, t-alpha, AR, tail behavior,
fraction-to-half, analytical and bootstrap SR standard errors,
plus a two-panel plot (cumulative P&L + drawdown).

In [ ]:
def Diagnostics(R, label='Strategy'):
    """
    Full diagnostic suite for a series of strategy returns (Lecture 11).
    R: pd.Series of per-trade or per-period net P&L.
    Returns a pd.Series of metrics and produces a two-panel plot.
    """
    results = {}
    T = len(R)

    sr_raw   = R.mean() / R.std(ddof=1)
    sr_ann   = sr_raw * ann_factor
    results['SR (annualised)']          = sr_ann
    results['Vol (annualised)']         = R.std(ddof=1) * ann_factor
    results['Mean return (annualised)'] = R.mean() * (252 / HOLD_PERIOD)
    results['t-stat (mean > 0)']        = R.mean() / (R.std(ddof=1) / np.sqrt(T))

    # CAPM alpha from daily series (if available)
    results['Alpha (annualised)']       = alpha_ann
    results['t-stat (alpha)']           = t_alpha
    results['Beta (Mkt-RF)']            = beta
    results['Appraisal ratio (AR)']     = ar_ann

    # Tail behavior: fraction of returns beyond ±3σ
    sigma = R.std(ddof=1)
    results['Tail fraction (|R|>3σ)']  = ((R < -3*sigma) | (R > 3*sigma)).mean()
    results['Min return']               = R.min()
    results['Max return']               = R.max()

    # SR standard errors
    se_analytical          = SR_vol(R) * ann_factor
    se_boot, sr_p05_boot   = SR_vol_boot(R)
    results['SR SE (analytical)']      = se_analytical
    results['SR SE (bootstrap)']       = se_boot * ann_factor
    results['SR 5th pct (bootstrap)']  = sr_p05_boot * ann_factor
    results['t-stat (SR)']             = sr_ann / se_analytical if se_analytical > 0 else 0

    # Fraction to half
    results['Fraction-to-half']        = fractiontohalf(R)

    # 95% CI for mean net P&L per trade
    t_crit = stats.t.ppf(0.975, df=T-1)
    se_mean = R.std(ddof=1) / np.sqrt(T)
    results['95% CI mean (lo)']        = R.mean() - t_crit * se_mean
    results['95% CI mean (hi)']        = R.mean() + t_crit * se_mean

    # Max drawdown (on cumulative P&L sequence)
    cum = R.cumsum()
    results['Max drawdown (cumul.)']   = (cum - cum.cummax()).min()

    # ── Two-panel plot: cumulative P&L + drawdown ─────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Panel 1: cumulative P&L
    ax = axes[0]
    ax.plot(cum.values, color='steelblue', linewidth=1.5, label=label)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.fill_between(range(len(cum)), cum.values, 0,
                    where=(cum.values >= 0), alpha=0.12, color='green')
    ax.fill_between(range(len(cum)), cum.values, 0,
                    where=(cum.values <  0), alpha=0.12, color='red')
    ax.set_title(f'{label}: Cumulative Net P&L', fontweight='bold')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Cumulative Net PnL (per $ bankroll)')
    ax.legend()

    # Panel 2: drawdown
    ax = axes[1]
    dd = cum - cum.cummax()
    ax.plot(dd.values, color='firebrick', linewidth=1)
    ax.fill_between(range(len(dd)), dd.values, 0, alpha=0.2, color='firebrick')
    ax.set_title(f'{label}: Drawdown', fontweight='bold')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Drawdown (cumulative P&L from peak)')

    plt.tight_layout()
    plt.show()

    return pd.Series(results)


# Run diagnostics on the full sample
full_diag = Diagnostics(net, label='Foundation Strategy (full sample)')
print(full_diag.to_frame('Full sample').to_string(float_format='{:+.4f}'.format))

### 6.5 Sample Splitting: Estimation vs. Hold-Out

From Lecture 11: *"A valid trading strategy can only use information known at the time of the trade.
Split the sample: estimate on the first half, test on the second. The hold-out result is the only
uncontaminated evidence."*

Here we split the trade sequence in half by time:
- **First half** (trades 1–N/2): in-sample estimation period
- **Second half** (trades N/2–N): hold-out test period — should not be used for parameter tuning

In [ ]:
mid = len(df) // 2
R_est  = net.iloc[:mid].reset_index(drop=True)
R_test = net.iloc[mid:].reset_index(drop=True)

print('═'*60)
print('  ESTIMATION SAMPLE (first half)')
print('═'*60)
est_diag = Diagnostics(R_est, label='Estimation half')

print('═'*60)
print('  HOLD-OUT / TEST SAMPLE (second half)')
print('═'*60)
test_diag = Diagnostics(R_test, label='Hold-out half')

# Side-by-side table
comparison = pd.concat([est_diag, test_diag], axis=1,
                        keys=['Estimation', 'Hold-out'])
print('\nSide-by-side comparison:')
print(comparison.to_string(float_format='{:+.4f}'.format))

### 6.6 Volatility Quintile Analysis (Lecture 5.7 mechanism check)

If vol-timing works as Lecture 5.7 predicts, low-RV quintiles should have better risk-adjusted returns.
**Empirically for prediction markets the relationship is inverted** — high-vol markets show the strongest
mean-reversion, so vol-timing actively hurts. γ=2 causes the weight cap to bind uniformly, which
empirically outperforms explicit vol-scaling here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_q = df.copy()
df_q['RV_Q'] = pd.qcut(df_q['RV'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
q_stats = df_q.groupby('RV_Q').agg(
    Avg_RV   = ('RV',      'mean'),
    Mean_Net = ('Net PnL', 'mean'),
    Win_Rate = ('Win',     lambda x: x.mean() * 100),
    N        = ('Ticker',  'count'),
).reset_index()

# Bar chart: mean net P&L per vol quintile
ax = axes[0]
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in q_stats['Mean_Net']]
bars = ax.bar(q_stats['RV_Q'].astype(str), q_stats['Mean_Net'],
              color=colors, alpha=0.75, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
for bar, (_, row) in zip(bars, q_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + abs(q_stats['Mean_Net']).max() * 0.03,
            f"{row['Win_Rate']:.0f}% win\n(n={int(row['N'])})",
            ha='center', va='bottom', fontsize=8)
ax.set_title('Mean Net P&L by Vol Quintile\n(Lecture 5.7: does lower vol → better returns?)',
             fontweight='bold')
ax.set_xlabel('Realized Variance Quintile (Q1=lowest)')
ax.set_ylabel('Mean Net PnL')

# Entry price split
ax = axes[1]
p40, p60 = df['Entry P'].quantile(0.40), df['Entry P'].quantile(0.60)
buckets = [
    (f'Low (P<{p40:.2f})',           df[df['Entry P'] <  p40]),
    (f'Mid ({p40:.2f}–{p60:.2f})',   df[(df['Entry P'] >= p40) & (df['Entry P'] < p60)]),
    (f'High (P≥{p60:.2f})',          df[df['Entry P'] >= p60]),
]
labels_ep = [b[0] for b in buckets]
means_ep  = [b[1]['Net PnL'].mean() for b in buckets]
wins_ep   = [b[1]['Win'].mean() * 100 for b in buckets]
ns_ep     = [len(b[1]) for b in buckets]
colors_ep = ['#2ca02c' if v >= 0 else '#d62728' for v in means_ep]
bars2 = ax.bar(labels_ep, means_ep, color=colors_ep, alpha=0.75,
               edgecolor='black', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
for bar, win, n_b in zip(bars2, wins_ep, ns_ep):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + abs(max(means_ep)) * 0.03,
            f'{win:.0f}% win\n(n={n_b})', ha='center', va='bottom', fontsize=8)
ax.set_title('Mean Net P&L by Entry Price Bucket\n(Key empirical finding)',
             fontweight='bold')
ax.set_ylabel('Mean Net PnL')

plt.tight_layout()
plt.savefig('vol_and_price_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.7 Adjusting for Multiple Testing

From Lecture 11: *"When you try many signals, conventional t-test thresholds are too lenient.
The Bonferroni correction: divide your significance level by the number of tests."*

Our strategy passed through several design decisions backed by observed data:
- Direction filter: tried momentum first (failed), then mean-reversion → **1 test**
- BUY NO only: observed that BUY YES had negative Sharpe → **1 test**
- Min-price filter: identified P < 0.15 losing sub-sample → **1 test**
- Sports exclusion: observed sports markets always lost → **1 test**
- Min-R² threshold: tried 0.25, 0.15, settled on 0.10 → **~3 tests**
- γ: tried γ=20 (worse), settled on γ=2 → **1 test**

Conservatively: **~8 data-driven decisions**. Bonferroni-adjusted threshold:

In [ ]:
n_tests = 8    # conservative count of data-driven design decisions
alpha_nominal  = 0.05
alpha_bonf     = alpha_nominal / n_tests
z_nominal      = norm.ppf(1 - alpha_nominal / 2)
z_bonferroni   = norm.ppf(1 - alpha_bonf / 2)

# Our observed t-stat (from mean net P&L test)
t_obs = net.mean() / (net.std(ddof=1) / np.sqrt(len(net)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bonferroni threshold vs number of tests
ax = axes[0]
n_range = np.arange(1, 51)
thresholds = norm.ppf(1 - alpha_nominal / (2 * n_range))
ax.plot(n_range, thresholds, linewidth=2, color='steelblue')
ax.axhline(z_nominal, color='grey', linestyle='--', linewidth=1,
           label=f'Naive (1 test): {z_nominal:.2f}')
ax.axhline(t_obs, color='red', linestyle='--', linewidth=1.5,
           label=f'Our t-stat: {t_obs:.2f}')
ax.axvline(n_tests, color='orange', linestyle=':', linewidth=1.2,
           label=f'Our n_tests = {n_tests} → z = {z_bonferroni:.2f}')
ax.set_xlabel('Number of tests (strategies tried)')
ax.set_ylabel('Required |t-stat| for 5% significance')
ax.set_title('Bonferroni Correction:\nRequired t-stat rises with # of tests', fontweight='bold')
ax.legend(fontsize=8)

# Right: Simulate noise strategies — how many pass the naive threshold?
ax = axes[1]
rng   = np.random.default_rng(99)
n_sim = 100
T_sim = len(net)
noise_returns = rng.normal(0, net.std(), (T_sim, n_sim))
t_noise = noise_returns.mean(axis=0) / (noise_returns.std(axis=0) / np.sqrt(T_sim))
ax.hist(t_noise, bins=20, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.4,
        label='Noise strategies (n=100)')
ax.axvline(z_nominal,    color='grey',   linestyle='--', linewidth=1.5,
           label=f'Naive 5% bar: {z_nominal:.2f}')
ax.axvline(z_bonferroni, color='orange', linestyle='--', linewidth=1.5,
           label=f'Bonferroni bar: {z_bonferroni:.2f}')
ax.axvline(t_obs,        color='red',    linestyle='-',  linewidth=2,
           label=f'Our t-stat: {t_obs:.2f}')
ax.set_xlabel('t-statistic')
ax.set_ylabel('Count')
ax.set_title(f'Pure-noise t-stats ({n_sim} strategies, T={T_sim})\nHow many pass by chance?',
             fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('multiple_testing.png', dpi=150, bbox_inches='tight')
plt.show()

n_false_positives = (t_noise > z_nominal).sum()
print(f'Naive threshold ({z_nominal:.2f}): {n_false_positives}/{n_sim} noise strategies pass')
print(f'Bonferroni threshold ({z_bonferroni:.2f}): {(t_noise > z_bonferroni).sum()}/{n_sim} noise strategies pass')
print(f'Our t-stat ({t_obs:.2f}) clears Bonferroni bar: {t_obs > z_bonferroni}')

### 6.8 Data Mining and Overfitting Discussion

From Lecture 11: *"Every sample estimate is a random variable. Optimization amplifies noise:
it selects signals that are pure noise, overweights real signals, and discards real signals
that did not show up strongly enough."*

**Risks specific to this strategy:**
1. **Survivorship bias** — only currently-open markets are backtested. Resolved markets
   (many of which may have been losers) are excluded by construction.
2. **Concentration** — in both backtest runs, two Texas Senate nomination markets contributed
   ~100% of total net P&L. The strategy may be capturing a specific market structure rather
   than a general phenomenon.
3. **Temporal instability** — first-half SR (+2.9) far exceeds second-half SR (+1.0),
   consistent with either signal decay or a lucky early period.
4. **Parameter tuning** — min-price=0.15 was chosen after observing that P<0.15 trades lost
   money, adding one degree of in-sample freedom.
5. **Small sample** — 58 trades. The bootstrap Sharpe CI is wide ([+0.01, +3.37]);
   the true Sharpe could be close to zero.

In [ ]:
# Visualise: what fraction of P&L comes from each market?
mkt_pnl = (df.groupby('Ticker')['Net PnL'].sum()
             .sort_values(ascending=False)
             .reset_index())
mkt_pnl['Cumulative share'] = mkt_pnl['Net PnL'].cumsum() / mkt_pnl['Net PnL'].sum() * 100
total_pnl = df['Net PnL'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: P&L by market (bar, top 15)
ax = axes[0]
top15 = mkt_pnl.head(15)
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in top15['Net PnL']]
ax.barh(top15['Ticker'], top15['Net PnL'], color=colors, alpha=0.75,
        edgecolor='black', linewidth=0.4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Total Net P&L by Market (top 15)\n⚠ Concentration risk', fontweight='bold')
ax.set_xlabel('Total Net PnL')
ax.invert_yaxis()

# Right: cumulative P&L share — "how many markets needed to get 100% of returns?"
ax = axes[1]
ax.plot(range(1, len(mkt_pnl) + 1), mkt_pnl['Cumulative share'],
        marker='o', markersize=4, linewidth=1.5, color='steelblue')
ax.axhline(100, color='grey', linestyle='--', linewidth=1)
ax.axhline(50,  color='grey', linestyle=':',  linewidth=1, label='50% of P&L')
# Mark how many markets it takes to reach 100%
n_to_100 = (mkt_pnl['Cumulative share'] <= 100).sum()
ax.axvline(2, color='red', linestyle='--', linewidth=1.5,
           label=f'Top 2 markets ≈ 100% of P&L')
ax.set_xlabel('Number of markets (ranked by P&L)')
ax.set_ylabel('Cumulative % of total net P&L')
ax.set_title('P&L Concentration:\nHow many markets generate all returns?', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('concentration_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

top2_pnl = mkt_pnl.head(2)['Net PnL'].sum()
print(f'Total net P&L:         {total_pnl:+.4f}')
print(f'Top 2 markets:         {top2_pnl:+.4f}  ({top2_pnl/total_pnl*100:.0f}% of total)')
print(f'Remaining {len(mkt_pnl)-2} markets: {total_pnl-top2_pnl:+.4f}  ({(total_pnl-top2_pnl)/total_pnl*100:.0f}% of total)')

---
## 7. Detailed Tables
### Table 1: Sub-Sample Analysis (Lecture 11: Section 11.10 Sample Splitting)

In [ ]:
def sub_stats(sub):
    n = len(sub)
    if n < 2:
        return n, 0.0, float('nan'), float('nan'), 0.0, 1.0
    net_s = sub['Net PnL']
    mn    = net_s.mean()
    sd    = net_s.std(ddof=1)
    sr    = (mn / sd) * ann_factor if sd > 1e-8 else 0.0
    t, p  = stats.ttest_1samp(net_s, 0)
    win   = sub['Win'].mean() * 100
    return n, win, mn, sr, t, p

def sig_stars(p):
    return '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else 'ns'))

mid = len(df) // 2
r2m = df['R2'].median()
p40, p60 = df['Entry P'].quantile(0.40), df['Entry P'].quantile(0.60)

sub_rows = []
for label, sub in [('Estimation (first half)', df.iloc[:mid]),
                   ('Hold-out  (second half)', df.iloc[mid:])]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

sub_rows.append(['─── Signal quality (R² split) ───', '', '', '', '', '', ''])
for label, sub in [(f'Low R²  (< {r2m:.3f})', df[df['R2'] < r2m]),
                   (f'High R² (≥ {r2m:.3f})', df[df['R2'] >= r2m])]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

sub_rows.append(['─── Entry price split ───', '', '', '', '', '', ''])
for label, sub in [
    (f'Low   entry (P < {p40:.2f})',            df[df['Entry P'] <  p40]),
    (f'Mid   entry ({p40:.2f}–{p60:.2f})',       df[(df['Entry P'] >= p40) & (df['Entry P'] < p60)]),
    (f'High  entry (P ≥ {p60:.2f})',             df[df['Entry P'] >= p60]),
]:
    n, win, mn, sr, t, p = sub_stats(sub)
    sub_rows.append([label, n, f'{win:.1f}%', f'{mn:+.5f}', f'{sr:+.3f}',
                     f'{t:+.2f}', f'{p:.4f} {sig_stars(p)}'])

print(tabulate(sub_rows,
               headers=['Sub-sample', 'N', 'Win%', 'Mean Net', 'Sharpe', 't-stat', 'p-val'],
               tablefmt='rounded_outline'))

### Table 2: Robustness Checks (Lecture 11: Section 11.9 Multiple Testing)

In [ ]:
rob_rows = []

rob_rows.append(['─── min-R² sensitivity ───', '', '', '', '', ''])
for thresh in [0.05, 0.10, 0.15, 0.20, 0.25]:
    sub = df[df['R2'] >= thresh]
    if len(sub) < 2:
        rob_rows.append([f'R² ≥ {thresh}', len(sub), '—', '—', '—', '(too few)'])
        continue
    n, win, mn, sr, t, p = sub_stats(sub)
    rob_rows.append([f'R² ≥ {thresh}', n, f'{win:.1f}%',
                     f'{mn:+.6f}', f'{sr:+.3f}', f'{p:.4f} {sig_stars(p)}'])

rob_rows.append(['─── Cost-layer sensitivity ───', '', '', '', '', ''])
raw_s         = df['Raw PnL']
after_spread  = raw_s - (raw_s - net - df['Kalshi Fee'])
for label, series in [
    ('Gross (no costs)',             raw_s),
    ('After spread only',            after_spread),
    ('Net (spread + Kalshi fee)',     net),
]:
    mn_s = series.mean()
    sd_s = series.std(ddof=1)
    sr_s = (mn_s / sd_s) * ann_factor if sd_s > 1e-8 else 0.0
    _, p_s = stats.ttest_1samp(series, 0)
    rob_rows.append([label, len(series), f'{series.gt(0).mean()*100:.1f}%',
                     f'{mn_s:+.6f}', f'{sr_s:+.3f}', f'{p_s:.4f} {sig_stars(p_s)}'])

rob_rows.append(['─── Leave-one-market-out ───', '', '', '', '', ''])
top_mkt   = df.groupby('Ticker')['Net PnL'].count().idxmax()
top_mkt_n = (df['Ticker'] == top_mkt).sum()
sub_loo   = df[df['Ticker'] != top_mkt]
if len(sub_loo) >= 2:
    n, win, mn, sr, t, p = sub_stats(sub_loo)
    rob_rows.append([f'Drop {top_mkt} ({top_mkt_n} trades)',
                     n, f'{win:.1f}%', f'{mn:+.6f}', f'{sr:+.3f}', f'{p:.4f} {sig_stars(p)}'])

print(tabulate(rob_rows,
               headers=['Check', 'N', 'Win%', 'Mean Net', 'Sharpe', 'p-val'],
               tablefmt='rounded_outline'))

### Table 3: Vol Quintile P&L  &  Table 4: P&L Percentiles

In [ ]:
# Vol quintile table
rows = []
for _, r in q_stats.iterrows():
    rows.append([r['RV_Q'], f"{r['Avg_RV']:.4f}", f"{np.sqrt(r['Avg_RV']):.1%}",
                 f"{r['Mean_Net']:+.6f}", f"{r['Win_Rate']:.1f}%", int(r['N'])])
print('Vol Quintile P&L:')
print(tabulate(rows, headers=['Q','Avg RV','Avg Vol','Mean Net PnL','Win %','N'],
               tablefmt='rounded_outline'))

# P&L percentiles
print('\nNet P&L Percentiles:')
pcts = [5, 10, 25, 50, 75, 90, 95]
print(tabulate([[f'p{p}', f'{v:+.6f}'] for p, v in zip(pcts, np.percentile(net, pcts))],
               headers=['Pct', 'Net PnL'], tablefmt='rounded_outline'))

### Table 5: Top Markets by Trade Count

In [ ]:
top_mkt_table = (
    df.groupby(['Ticker', 'Title'])
    .agg(Trades=('Win','count'), Win_Rate=('Win', lambda x: x.mean()*100),
         Mean_Net=('Net PnL','mean'), Total_Net=('Net PnL','sum'))
    .sort_values('Trades', ascending=False).head(20).reset_index()
)
rows = [[r['Ticker'], r['Title'][:36], int(r['Trades']),
         f"{r['Win_Rate']:.1f}%", f"{r['Mean_Net']:+.6f}", f"{r['Total_Net']:+.6f}"]
        for _, r in top_mkt_table.iterrows()]
print(tabulate(rows, headers=['Ticker','Title','Trades','Win %','Mean Net','Total Net'],
               tablefmt='rounded_outline'))

print(f'\nTotal net P&L:  {total_pnl:+.4f}')
print(f'Top 2 markets:  {top2_pnl:+.4f}  ({top2_pnl/total_pnl*100:.0f}% of total)')
print('\n⚠  Survivorship bias: only currently-open markets are backtested.')
print(f'   Costs: half-spread×|w|×entry  +  Kalshi fee 0.07×|w|×entry.')
print(f'   Annualized Sharpe assumes {int(252/HOLD_PERIOD)} non-overlapping periods/year.')